# Derive Sonar Submission Stats

This notebook derives the headline numbers used in the Sonar-submission results paragraph. It reads `lidskjalv_invocations.csv` (the pipeline's build/submit log), `sonar_resubmissions.csv` (the manual recovery pass), `andvari_invocations.csv` (the upstream step whose runner-timeout errors blocked three submissions downstream), and the `project_key` column of `sonar_projects.csv` (used only to check whether a resubmission produced a metrics row). The notebook prints every figure quoted in the paragraph and asserts on each; it does not write any derived CSV.

In [ ]:
from pathlib import Path

import pandas as pd


def find_package_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'data' / 'exported' / 'lidskjalv_invocations.csv').exists():
            return path
    raise FileNotFoundError('Could not find data/exported/lidskjalv_invocations.csv')


PACKAGE_ROOT = find_package_root(Path.cwd())
EXPORTED = PACKAGE_ROOT / 'data' / 'exported'

lidskjalv = pd.read_csv(EXPORTED / 'lidskjalv_invocations.csv')
resubmissions = pd.read_csv(EXPORTED / 'sonar_resubmissions.csv')
andvari = pd.read_csv(EXPORTED / 'andvari_invocations.csv')
sonar_projects = pd.read_csv(EXPORTED / 'sonar_projects.csv')

lidskjalv.shape, resubmissions.shape, andvari.shape, sonar_projects.shape

## Validate Required Columns

Each downstream step depends on a small, stable subset of each table; missing columns indicate the export schema drifted.

In [ ]:
required = {
    'lidskjalv_invocations.csv': (lidskjalv, [
        'agent', 'repo_slug', 'variant', 'status', 'reason', 'report_status',
        'input_repo_source', 'duration_seconds',
    ]),
    'sonar_resubmissions.csv': (resubmissions, [
        'repo_slug', 'agent', 'variant', 'project_key', 'latest_submission_success',
    ]),
    'andvari_invocations.csv': (andvari, [
        'repo_slug', 'variant', 'agent', 'status', 'reason',
    ]),
    'sonar_projects.csv': (sonar_projects, ['project_key']),
}

for name, (df, cols) in required.items():
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f'{name}: missing columns {missing}')

## Normalise Resubmission Flags

`latest_submission_success` is stored as a string (`True` / `False`); cast it to a canonical boolean.

In [ ]:
def _as_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower().eq('true')


resubmissions['submission_worked'] = _as_bool(resubmissions['latest_submission_success'])

## Submission Report Status Counts

The codex and claude runs of each original repo submit to the same `<repo>__original` project, so they're collapsed per repo. That gives 280 deduplicated submission targets (40 original + 80 each for the generated, v2, and v3 variants). The cell below derives:

- how many Lidskjálv invocations reported `passed`, `failed`, or produced no report;
- how the target-level report-status distribution breaks down by variant, with original repos deduplicated;
- how many no-report cases were skipped because Andvari timed out upstream, and how many were Lidskjálv timeouts;
- how many resubmission jobs ran, whether they all reported `submission_success`, and how many were hollow (no metrics row was produced for the exact project key);
- the Lidskjálv `input_repo_source` distribution and median runtime in `duration_seconds`.

In [ ]:
VARIANT_ORDER = ['original', 'generated', 'v2', 'v3']
REPORT_ORDER = ['passed', 'failed', 'no_report']
DEDUP_TARGETS = {'original': 40, 'generated': 80, 'v2': 80, 'v3': 80}
total_targets = sum(DEDUP_TARGETS.values())
raw_invocations = int(len(lidskjalv))

# Lidskjálv report-status summaries. Empty report_status means no report was produced.
lidskjalv['report_bucket'] = lidskjalv['report_status'].fillna('no_report').replace('', 'no_report')
raw_report_status_counts = {
    status: int((lidskjalv['report_bucket'] == status).sum())
    for status in REPORT_ORDER
}


def dedup_key(row):
    if row['variant'] == 'original':
        return f"{row['repo_slug']}__original"
    return f"{row['repo_slug']}__{row['variant']}__{row['agent']}"


def target_report_bucket(group):
    buckets = set(group['report_bucket'])
    if 'passed' in buckets:
        return 'passed'
    if 'failed' in buckets:
        return 'failed'
    return 'no_report'


# Target-level records collapse the codex/claude original scans to one project.
lidskjalv['dedup_key'] = lidskjalv.apply(dedup_key, axis=1)
target_records = []
for target_key, group in lidskjalv.groupby('dedup_key', sort=False):
    target_records.append({
        'target_key': target_key,
        'variant': group['variant'].iloc[0],
        'report_bucket': target_report_bucket(group),
        # For deduplicated originals, use the median of the paired codex/claude runtimes.
        'duration_seconds': float(group['duration_seconds'].median()),
    })
target_lidskjalv = pd.DataFrame(target_records)
target_report_counts = {
    status: int((target_lidskjalv['report_bucket'] == status).sum())
    for status in REPORT_ORDER
}
target_report_by_variant = (
    target_lidskjalv.pivot_table(
        index='variant', columns='report_bucket', values='target_key',
        aggfunc='count', fill_value=0,
    )
    .reindex(index=VARIANT_ORDER, columns=REPORT_ORDER, fill_value=0)
    .astype(int)
)
target_median_runtime_seconds = target_lidskjalv.groupby('variant')['duration_seconds'].median().reindex(VARIANT_ORDER)
variant_table = target_report_by_variant.copy()
variant_table['total'] = variant_table[REPORT_ORDER].sum(axis=1)
variant_table['median_runtime_seconds'] = target_median_runtime_seconds
variant_table['median_runtime_minutes'] = target_median_runtime_seconds / 60
total_row = pd.DataFrame([{
    **target_report_counts,
    'total': total_targets,
    'median_runtime_seconds': float(target_lidskjalv['duration_seconds'].median()),
    'median_runtime_minutes': float(target_lidskjalv['duration_seconds'].median() / 60),
}], index=['total'])
variant_table_with_total = pd.concat([variant_table, total_row])

# Lidskjálv input source and runtime summaries
input_repo_sources = {
    source: int(count)
    for source, count in lidskjalv['input_repo_source'].value_counts().to_dict().items()
}
median_duration_seconds = float(lidskjalv['duration_seconds'].median())
median_duration_minutes = median_duration_seconds / 60

# No-report cases: upstream skips vs. Lidskjálv's own timeout errors
raw_no_report_rows = lidskjalv[lidskjalv['report_bucket'] == 'no_report']
raw_no_report_reason_counts = raw_no_report_rows.groupby(['status', 'reason']).size().to_dict()
andvari_blocked_no_report_invocations = int((
    (raw_no_report_rows['status'] == 'blocked')
    & (raw_no_report_rows['reason'] == 'blocked-by-upstream')
    & (raw_no_report_rows['input_repo_source'] == 'andvari')
).sum())
lidskjalv_timeout_no_report_invocations = int((
    (raw_no_report_rows['status'] == 'error')
    & (raw_no_report_rows['reason'] == 'lidskjalv-timeout')
).sum())
failed_reason_counts = (
    lidskjalv.loc[lidskjalv['report_bucket'] == 'failed', 'reason']
    .fillna('unknown')
    .replace('', 'unknown')
    .value_counts()
    .to_dict()
)

# Manual resubmission outcomes and final Sonar metrics coverage
sonar_keys = set(sonar_projects['project_key'])
resubmissions['landed_in_metrics'] = resubmissions['project_key'].isin(sonar_keys)
n_jobs = int(len(resubmissions))
all_reported_success = bool(resubmissions['submission_worked'].all())
hollow = int((~resubmissions['landed_in_metrics']).sum())
metrics_rows = int(sonar_projects['project_key'].nunique())
final_missing = total_targets - metrics_rows

print(f'Deduplicated submission targets      : {total_targets}')
print(f'  by variant                         : {DEDUP_TARGETS}')
print(f'Lidskjálv invocations                : {raw_invocations}')
print(f'Raw invocation report status counts  : {raw_report_status_counts}')
print(f'Deduplicated target report statuses : {target_report_counts}')
print()
print('Target report status by variant:')
print(variant_table_with_total.to_string(formatters={
    'median_runtime_seconds': '{:.2f}'.format,
    'median_runtime_minutes': '{:.1f}'.format,
}))
print()
print(f'Input repo source counts             : {input_repo_sources}')
print(f'Median runtime seconds               : {median_duration_seconds:g}')
print(f'Median runtime minutes               : {median_duration_minutes:.1f}')
print()
print(f'Raw no-report invocations            : {raw_report_status_counts["no_report"]}')
print(f'  blocked by upstream Andvari failure: {andvari_blocked_no_report_invocations}')
print(f'  Lidskjálv timeout errors           : {lidskjalv_timeout_no_report_invocations}')
print(f'  status/reason breakdown            : {raw_no_report_reason_counts}')
print(f'Failed report reasons                : {failed_reason_counts}')
print()
print(f'Manual resubmission jobs             : {n_jobs}')
print(f'  all reported submission_success    : {all_reported_success}')
print(f'  hollow (no metrics row produced)   : {hollow}')
print(f'Final Sonar metrics rows             : {metrics_rows}/{total_targets}')
print(f'Remaining gaps after full pipeline   : {final_missing}')

assert total_targets == 280
assert raw_invocations == 320
assert len(target_lidskjalv) == total_targets
assert raw_report_status_counts == {'passed': 273, 'failed': 41, 'no_report': 6}
assert target_report_counts == {'passed': 252, 'failed': 24, 'no_report': 4}
assert target_report_by_variant.to_dict(orient='index') == {
    'original': {'passed': 21, 'failed': 18, 'no_report': 1},
    'generated': {'passed': 78, 'failed': 0, 'no_report': 2},
    'v2': {'passed': 78, 'failed': 2, 'no_report': 0},
    'v3': {'passed': 75, 'failed': 4, 'no_report': 1},
}
assert variant_table['total'].to_dict() == {'original': 40, 'generated': 80, 'v2': 80, 'v3': 80}
assert variant_table['median_runtime_seconds'].to_dict() == {
    'original': 182.75,
    'generated': 167.0,
    'v2': 166.0,
    'v3': 166.0,
}
assert input_repo_sources == {'kvasir_ported': 228, 'brokk_original': 80, 'andvari': 12}
assert median_duration_seconds == 168.5
assert andvari_blocked_no_report_invocations == 3
assert lidskjalv_timeout_no_report_invocations == 3
assert n_jobs == 32 and all_reported_success
assert hollow == 3
assert metrics_rows == 277
assert final_missing == 3